# Mean Anomaly Shift Test — S3M NEOs

Take the S3M NEO catalog and create **36 copies**, each with every object's mean anomaly shifted by an extra **k × 10°** (k = 0 … 35).  
This rotates the whole population along its orbits in 10° steps through one full revolution.

**Part A — raw shifts:** just change `t_p` to encode the new M, compute heliocentric ecliptic x, y, plot.  
**Part B — K|M cloner applied to each shifted population:** feed each shifted catalog into `clone_population_conditional_K_from_M` and see how the cloner output changes.

Reuses pipeline functions from `velocity_density_pipeline.py` and `neoscore.py`.

## 0. Setup

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from astropy.time import Time

# In a notebook, cwd is where the .ipynb lives (neo_mean_anomaly_shift_test/).
# S3M loader searches for data relative to cwd, so we must be in neomod/.
TESTDIR = os.getcwd()                            # .../neomod/neo_mean_anomaly_shift_test
NEOMOD  = os.path.abspath(os.path.join(TESTDIR, '..'))   # .../neomod

os.chdir(NEOMOD)                                 # so S3M loader finds S3Mdata/
if NEOMOD not in sys.path:
    sys.path.insert(0, NEOMOD)

import velocity_density_pipeline as vdp
import neoscore as nsc

AU_KM        = 149_597_870.7
OBSTIME_STR  = "2025-03-21T00:00:00"
T_EPOCH_MJD  = Time(OBSTIME_STR, scale='tdb').mjd
N_SHIFTS     = 36
DELTA_DEG    = 10.0
N_SUBSAMPLE  = 50_000
N_PLOT       = 5_000
CLONE_FACTOR = 5

def figpath(name):
    """Save figures into the test folder, not neomod/."""
    return os.path.join(TESTDIR, name)

print(f"NEOMOD  : {NEOMOD}")
print(f"TESTDIR : {TESTDIR}")
print(f"Epoch MJD: {T_EPOCH_MJD:.2f}")
print(f"36 shifts × {DELTA_DEG}° = one full orbit")

## 1. Load S3M NEO population

In [ ]:
df_full, scorer = vdp.load_s3m_population('neo')
print(f"Loaded {len(df_full):,} NEOs")
print("Columns:", df_full.columns.tolist())

# Subsample for speed
rng = np.random.default_rng(42)
if len(df_full) > N_SUBSAMPLE:
    idx = rng.choice(len(df_full), N_SUBSAMPLE, replace=False)
    df_neo = df_full.iloc[idx].reset_index(drop=True)
else:
    df_neo = df_full.reset_index(drop=True)

print(f"Using {len(df_neo):,} NEOs for this test")

## 2. Helper functions

In [ ]:
def shift_mean_anomaly(df, delta_deg, t_epoch_mjd=T_EPOCH_MJD):
    """Return a copy of df with t_p modified so M_new = M_orig + delta_deg.

    Works by:
      1. Computing M_orig at the observation epoch from t_p and semi-major axis.
      2. Adding delta_deg (mod 360).
      3. Solving back for the new t_p that corresponds to M_new.
    """
    a   = df['a'].to_numpy(float)
    t_p = df['t_p'].to_numpy(float)

    # Mean motion in rad/day  (a in AU, T = a^1.5 years)
    n_rad_day = 2.0 * np.pi / (a**1.5 * 365.25)

    # Original mean anomaly at epoch
    M_orig = np.mod(n_rad_day * (t_epoch_mjd - t_p), 2.0 * np.pi)

    # Shifted mean anomaly
    M_new  = np.mod(M_orig + np.deg2rad(delta_deg), 2.0 * np.pi)

    # New periapsis passage time
    tp_new = t_epoch_mjd - M_new / n_rad_day

    out = df.copy()
    out['t_p'] = tp_new
    return out


def compute_helio_xy_au(df, obstime_str=OBSTIME_STR, chunk_size=20_000):
    """Return heliocentric ecliptic x, y in AU via elements_to_helio_ecliptic_state."""
    n = len(df)
    x_au = np.empty(n)
    y_au = np.empty(n)

    for start in range(0, n, chunk_size):
        cdf = df.iloc[start : start + chunk_size]
        r_ecl, _ = nsc.elements_to_helio_ecliptic_state(
            a_AU        = cdf['a'].to_numpy(float),
            e           = cdf['e'].to_numpy(float),
            inc_deg     = cdf['i'].to_numpy(float),
            raan_deg    = cdf['node'].to_numpy(float),
            argp_deg    = cdf['argperi'].to_numpy(float),
            tp_mjd      = cdf['t_p'].to_numpy(float),
            obstime_str = obstime_str,
            method='newton', n_iter=10,
            chunk=chunk_size, show_progress=False,
        )
        sl = slice(start, start + len(cdf))
        x_au[sl] = r_ecl[:, 0] / AU_KM
        y_au[sl] = r_ecl[:, 1] / AU_KM

    return x_au, y_au


def subsample_xy(x, y, n=N_PLOT, seed=0):
    """Return a random subsample for plotting (avoids overplotting)."""
    rng2 = np.random.default_rng(seed)
    idx  = rng2.choice(len(x), min(n, len(x)), replace=False)
    return x[idx], y[idx]


print("Helper functions defined.")

## Part A — Raw mean anomaly shifts

For each k = 0 … 35, shift every NEO's mean anomaly by k × 10° and compute its heliocentric ecliptic (x, y).  
No cloner — this is the exact S3M population at different orbital phases.

In [ ]:
# Pre-compute x,y for all 36 shifts (takes a few minutes)
xy_raw = {}   # k -> (x_au, y_au)

for k in range(N_SHIFTS):
    delta = k * DELTA_DEG
    df_shifted = shift_mean_anomaly(df_neo, delta)
    x, y = compute_helio_xy_au(df_shifted)
    xy_raw[k] = (x, y)
    if k % 6 == 0 or k == N_SHIFTS - 1:
        print(f"  k={k:2d}  Δ={delta:5.1f}°  done")

print(f"\nAll {N_SHIFTS} raw shifts computed.")

### A1. Side-by-side: selected shifts

In [ ]:
selected = [0, 1, 3, 6, 9, 17]
labels   = [f"Δ={k*10}°" for k in selected]

fig, axes = plt.subplots(2, 3, figsize=(14, 9), sharex=True, sharey=True)
lim = 4.5

for ax, k, lbl in zip(axes.flat, selected, labels):
    x, y = subsample_xy(*xy_raw[k])
    ax.scatter(x, y, s=1, alpha=0.25, color='royalblue', rasterized=True)
    th = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.5)
    ax.scatter([0], [0], color='gold', s=60, zorder=5, marker='*')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel('x (AU)'); ax.set_ylabel('y (AU)')

fig.suptitle('Part A — Heliocentric XY: raw M shifts (S3M NEOs)', fontsize=13)
plt.tight_layout()
plt.savefig(figpath('raw_shift_selected.png'), dpi=120, bbox_inches='tight')

### A2. 6 × 6 grid — all 36 shifts

In [ ]:
fig, axes = plt.subplots(6, 6, figsize=(18, 18), sharex=True, sharey=True)
lim = 4.5
th  = np.linspace(0, 2*np.pi, 200)

for k, ax in enumerate(axes.flat):
    x, y = subsample_xy(*xy_raw[k], n=2000)
    ax.scatter(x, y, s=0.6, alpha=0.3, color='royalblue', rasterized=True)
    ax.plot(np.cos(th), np.sin(th), 'g-', lw=0.5, alpha=0.4)
    ax.scatter([0], [0], color='gold', s=30, zorder=5, marker='*')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(f"k={k}  Δ={k*10}°", fontsize=7)
    ax.tick_params(labelsize=6)

fig.suptitle('Part A — All 36 raw M shifts (heliocentric ecliptic XY)', fontsize=14, y=1.002)
plt.tight_layout()
plt.savefig(figpath('raw_shift_grid_6x6.png'), dpi=100, bbox_inches='tight')

### A3. Animation — stepping through all 36 shifts

In [ ]:
fig_anim, ax_anim = plt.subplots(figsize=(6, 6))
lim = 4.5
th  = np.linspace(0, 2*np.pi, 200)

ax_anim.set_xlim(-lim, lim); ax_anim.set_ylim(-lim, lim)
ax_anim.set_aspect('equal')
ax_anim.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.5, label='Earth orbit')
ax_anim.scatter([0], [0], color='gold', s=80, zorder=5, marker='*', label='Sun')
ax_anim.set_xlabel('x (AU)'); ax_anim.set_ylabel('y (AU)')

# Pre-subsample for animation speed
anim_xy = [subsample_xy(*xy_raw[k], n=3000, seed=k) for k in range(N_SHIFTS)]

scat = ax_anim.scatter([], [], s=1.5, alpha=0.3, color='royalblue', rasterized=True)
title = ax_anim.set_title('')

def _update(frame):
    x, y = anim_xy[frame]
    scat.set_offsets(np.c_[x, y])
    title.set_text(f'k={frame}   ΔM = {frame * 10}°')
    return scat, title

anim_raw = animation.FuncAnimation(
    fig_anim, _update, frames=N_SHIFTS, interval=200, blit=True
)

plt.close(fig_anim)   # don't show static
HTML(anim_raw.to_jshtml())

## Part B — K|M cloner applied to each shifted population

Feed each of the 36 shifted catalogs into `clone_population_conditional_K_from_M`.  
The cloner resamples M from the empirical distribution of the input — so a shifted input M distribution should produce shifted output M values.  
The question: does the cloner faithfully reproduce the spatial shift, or does it wash it out?

We run the cloner for **all 36 steps** but with a small `clone_factor` (= 5) to keep runtime reasonable.

In [ ]:
def clone_to_df(shifted_df, clone_factor, obstime_str, rng):
    """Call the K|M cloner and reassemble the tuple result into a DataFrame."""
    a, e, inc, raan, argp, tp = vdp.clone_population_conditional_K_from_M(
        shifted_df,
        clone_factor = clone_factor,
        obstime_str  = obstime_str,
        rng          = rng,
    )
    return pd.DataFrame({'a': a, 'e': e, 'i': inc, 'node': raan, 'argperi': argp, 't_p': tp})


xy_cloned = {}   # k -> (x_au, y_au)

for k in range(N_SHIFTS):
    delta      = k * DELTA_DEG
    df_shifted = shift_mean_anomaly(df_neo, delta)

    df_cl = clone_to_df(df_shifted, CLONE_FACTOR, OBSTIME_STR, np.random.default_rng(k))

    x, y = compute_helio_xy_au(df_cl)
    xy_cloned[k] = (x, y)

    if k % 6 == 0 or k == N_SHIFTS - 1:
        print(f"  k={k:2d}  Δ={delta:5.1f}°  cloned {len(df_cl):,} → done")

print(f"\nAll {N_SHIFTS} cloned populations computed.")

### B1. Side-by-side: clone 1 (Δ=0°) vs clone 2 (Δ=10°) vs others

In [ ]:
selected = [0, 1, 3, 6, 9, 17]
labels   = [f"clone k={k}  Δ={k*10}°" for k in selected]

fig, axes = plt.subplots(2, 3, figsize=(14, 9), sharex=True, sharey=True)
lim = 4.5
th  = np.linspace(0, 2*np.pi, 300)

for ax, k, lbl in zip(axes.flat, selected, labels):
    x, y = subsample_xy(*xy_cloned[k])
    ax.scatter(x, y, s=1, alpha=0.25, color='darkorange', rasterized=True)
    ax.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.5)
    ax.scatter([0], [0], color='gold', s=60, zorder=5, marker='*')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel('x (AU)'); ax.set_ylabel('y (AU)')

fig.suptitle('Part B — Heliocentric XY: K|M cloned populations (shifted M inputs)', fontsize=12)
plt.tight_layout()
plt.savefig(figpath('cloned_shift_selected.png'), dpi=120, bbox_inches='tight')

### B2. 6 × 6 grid — cloned populations for all 36 shifts

In [ ]:
fig, axes = plt.subplots(6, 6, figsize=(18, 18), sharex=True, sharey=True)
lim = 4.5
th  = np.linspace(0, 2*np.pi, 200)

for k, ax in enumerate(axes.flat):
    x, y = subsample_xy(*xy_cloned[k], n=2000)
    ax.scatter(x, y, s=0.6, alpha=0.3, color='darkorange', rasterized=True)
    ax.plot(np.cos(th), np.sin(th), 'g-', lw=0.5, alpha=0.4)
    ax.scatter([0], [0], color='gold', s=30, zorder=5, marker='*')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(f"k={k}  Δ={k*10}°", fontsize=7)
    ax.tick_params(labelsize=6)

fig.suptitle('Part B — All 36 K|M cloned populations (shifted M inputs)', fontsize=14, y=1.002)
plt.tight_layout()
plt.savefig(figpath('cloned_shift_grid_6x6.png'), dpi=100, bbox_inches='tight')

### B3. Animation — cloned populations cycling through all 36 shifts

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(6, 6))
lim = 4.5
th  = np.linspace(0, 2*np.pi, 200)

ax_b.set_xlim(-lim, lim); ax_b.set_ylim(-lim, lim)
ax_b.set_aspect('equal')
ax_b.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.5, label='Earth orbit')
ax_b.scatter([0], [0], color='gold', s=80, zorder=5, marker='*', label='Sun')
ax_b.set_xlabel('x (AU)'); ax_b.set_ylabel('y (AU)')

anim_xy_cl = [subsample_xy(*xy_cloned[k], n=3000, seed=k+100) for k in range(N_SHIFTS)]

scat_b  = ax_b.scatter([], [], s=1.5, alpha=0.3, color='darkorange', rasterized=True)
title_b = ax_b.set_title('')

def _update_b(frame):
    x, y = anim_xy_cl[frame]
    scat_b.set_offsets(np.c_[x, y])
    title_b.set_text(f'K|M cloned  k={frame}   ΔM = {frame * 10}°')
    return scat_b, title_b

anim_cloned = animation.FuncAnimation(
    fig_b, _update_b, frames=N_SHIFTS, interval=200, blit=True
)

plt.close(fig_b)
HTML(anim_cloned.to_jshtml())

### B4. Direct overlay: raw shift vs cloned shift for the same k

In [ ]:
compare_k = [0, 1, 6, 18]

fig, axes = plt.subplots(len(compare_k), 2, figsize=(11, 4*len(compare_k)),
                         sharex=True, sharey=True)
lim = 4.5
th  = np.linspace(0, 2*np.pi, 300)

for row, k in enumerate(compare_k):
    for col, (xy_dict, color, label) in enumerate([
        (xy_raw,    'royalblue',  'raw shift'),
        (xy_cloned, 'darkorange', 'K|M cloned'),
    ]):
        ax = axes[row, col]
        x, y = subsample_xy(*xy_dict[k])
        ax.scatter(x, y, s=1.2, alpha=0.25, color=color, rasterized=True)
        ax.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.4)
        ax.scatter([0], [0], color='gold', s=50, zorder=5, marker='*')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect('equal')
        ax.set_title(f"{label}  k={k}  Δ={k*10}°", fontsize=10)
        ax.set_xlabel('x (AU)'); ax.set_ylabel('y (AU)')

fig.suptitle('Raw shift (left) vs K|M cloned (right) at the same ΔM', fontsize=12)
plt.tight_layout()
plt.savefig(figpath('raw_vs_cloned_overlay.png'), dpi=120, bbox_inches='tight')

## Part C — M distribution check

Quick sanity check: plot the M distribution at input and after cloning for k = 0, 1, 9, 18 to confirm the shift is actually being encoded and preserved.

In [ ]:
check_k = [0, 1, 9, 18]
fig, axes = plt.subplots(len(check_k), 2, figsize=(12, 3.5*len(check_k)))

for row, k in enumerate(check_k):
    delta = k * DELTA_DEG
    df_sh = shift_mean_anomaly(df_neo, delta)

    # M of the raw shifted input
    n_sh  = 2.0 * np.pi / (df_sh['a'].to_numpy(float)**1.5 * 365.25)
    M_sh  = np.mod(n_sh * (T_EPOCH_MJD - df_sh['t_p'].to_numpy(float)), 2*np.pi)

    # M of the cloned output — clone_to_df defined in Part B cell
    df_cl = clone_to_df(df_sh, CLONE_FACTOR, OBSTIME_STR, np.random.default_rng(k))
    n_cl  = 2.0 * np.pi / (df_cl['a'].to_numpy(float)**1.5 * 365.25)
    M_cl  = np.mod(n_cl * (T_EPOCH_MJD - df_cl['t_p'].to_numpy(float)), 2*np.pi)

    for ax, M, color, lbl in [
        (axes[row, 0], M_sh, 'royalblue',  f'raw input  k={k}  Δ={delta:.0f}°'),
        (axes[row, 1], M_cl, 'darkorange', f'K|M cloned k={k}  Δ={delta:.0f}°'),
    ]:
        ax.hist(np.rad2deg(M), bins=np.linspace(0, 360, 37),
                color=color, alpha=0.7, edgecolor='none')
        ax.set_xlabel('M (deg)'); ax.set_ylabel('count')
        ax.set_title(lbl, fontsize=10)
        ax.set_xlim(0, 360)

fig.suptitle('Mean anomaly distributions: raw shifted input vs K|M cloned output', fontsize=12)
plt.tight_layout()
plt.savefig(figpath('M_distribution_check.png'), dpi=120, bbox_inches='tight')

## Part D — Large-factor cloning (clone_factor = 1000)

Repeat Part B and Part C with `clone_factor = 1000` instead of 5.

**Scientific question:** with many more clones, does the K|M cloner converge to the same M distribution regardless of the shift (because NEO M is already ~uniform), or does the larger sample better resolve any residual non-uniformity?

**Computational note:** 50k NEOs × 1000 clones = 50M objects — too slow to propagate for all 36 shifts. We use a smaller base (`N_LARGE_BASE = 500`) so each shift produces 500k clones. M distributions are computed analytically (no propagation); x,y is computed only for selected k.

In [ ]:
LARGE_CLONE_FACTOR = 1000
N_LARGE_BASE = 500

rng_d = np.random.default_rng(99)
idx_d = rng_d.choice(len(df_neo), N_LARGE_BASE, replace=False)
df_large_base = df_neo.iloc[idx_d].reset_index(drop=True)

print(f"Part D base: {len(df_large_base):,} NEOs  ×  clone_factor={LARGE_CLONE_FACTOR}  →  {len(df_large_base)*LARGE_CLONE_FACTOR:,} clones per shift")

### D1. M distribution check — clone_factor = 1000

Same plot as Part C but with 1000× clones. Expect much smoother histograms; key question is whether the cloned M still tracks the (shifted) input shape.

In [ ]:
check_k = [0, 1, 9, 18]
fig, axes = plt.subplots(len(check_k), 2, figsize=(12, 3.5*len(check_k)))

for row, k in enumerate(check_k):
    delta = k * DELTA_DEG
    df_sh = shift_mean_anomaly(df_large_base, delta)

    # M of the raw shifted input (500 objects)
    n_sh = 2.0 * np.pi / (df_sh['a'].to_numpy(float)**1.5 * 365.25)
    M_sh = np.mod(n_sh * (T_EPOCH_MJD - df_sh['t_p'].to_numpy(float)), 2*np.pi)

    # M of the large cloned output (500,000 objects)
    df_cl = clone_to_df(df_sh, LARGE_CLONE_FACTOR, OBSTIME_STR, np.random.default_rng(k))
    n_cl  = 2.0 * np.pi / (df_cl['a'].to_numpy(float)**1.5 * 365.25)
    M_cl  = np.mod(n_cl * (T_EPOCH_MJD - df_cl['t_p'].to_numpy(float)), 2*np.pi)

    for ax, M, color, lbl, density in [
        (axes[row, 0], M_sh, 'royalblue',  f'raw input  k={k}  Δ={delta:.0f}°  (n={len(M_sh):,})',  False),
        (axes[row, 1], M_cl, 'darkorange', f'K|M cloned k={k}  Δ={delta:.0f}°  (n={len(M_cl):,})', True),
    ]:
        ax.hist(np.rad2deg(M), bins=np.linspace(0, 360, 37),
                color=color, alpha=0.7, edgecolor='none', density=density)
        ax.set_xlabel('M (deg)')
        ax.set_ylabel('density' if density else 'count')
        ax.set_title(lbl, fontsize=10)
        ax.set_xlim(0, 360)

fig.suptitle(f'Part D — M distributions: clone_factor={LARGE_CLONE_FACTOR}  (base n={N_LARGE_BASE})', fontsize=12)
plt.tight_layout()
plt.savefig(figpath('large_clone_M_distribution.png'), dpi=120, bbox_inches='tight')
print("Saved large_clone_M_distribution.png")

### D2. Heliocentric XY — clone_factor = 1000, selected shifts

Compute x,y for k = 0, 1, 9, 18 only (propagating 500k objects each). Side-by-side: raw shift (blue) vs large-clone (orange).

In [ ]:
selected_d = [0, 1, 9, 18]

# Raw x,y from the smaller base (no cloning, direct propagation)
xy_raw_large_base = {}
for k in selected_d:
    df_sh = shift_mean_anomaly(df_large_base, k * DELTA_DEG)
    xy_raw_large_base[k] = compute_helio_xy_au(df_sh)
    print(f"  raw k={k} done")

# Large-clone x,y
xy_large_clone = {}
for k in selected_d:
    df_sh = shift_mean_anomaly(df_large_base, k * DELTA_DEG)
    df_cl = clone_to_df(df_sh, LARGE_CLONE_FACTOR, OBSTIME_STR, np.random.default_rng(k+200))
    xy_large_clone[k] = compute_helio_xy_au(df_cl)
    print(f"  cloned k={k}  ({len(df_cl):,} objects) done")

print("All D2 propagations complete.")

In [ ]:
fig, axes = plt.subplots(len(selected_d), 2, figsize=(11, 4.5*len(selected_d)),
                         sharex=True, sharey=True)
lim = 4.5
th  = np.linspace(0, 2*np.pi, 300)

for row, k in enumerate(selected_d):
    for col, (xy_dict, color, label) in enumerate([
        (xy_raw_large_base, 'royalblue',  f'raw shift (n={N_LARGE_BASE:,})'),
        (xy_large_clone,    'darkorange', f'K|M cloned ×{LARGE_CLONE_FACTOR} (n={N_LARGE_BASE*LARGE_CLONE_FACTOR:,})'),
    ]):
        ax = axes[row, col]
        x, y = subsample_xy(*xy_dict[k], n=N_PLOT)
        ax.scatter(x, y, s=1.2, alpha=0.2, color=color, rasterized=True)
        ax.plot(np.cos(th), np.sin(th), 'g--', lw=0.8, alpha=0.4)
        ax.scatter([0], [0], color='gold', s=50, zorder=5, marker='*')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect('equal')
        ax.set_title(f"{label}  k={k}  Δ={k*10}°", fontsize=10)
        ax.set_xlabel('x (AU)'); ax.set_ylabel('y (AU)')

fig.suptitle(f'Part D — Raw vs K|M cloned ×{LARGE_CLONE_FACTOR} (heliocentric XY)', fontsize=12)
plt.tight_layout()
plt.savefig(figpath('large_clone_xy_comparison.png'), dpi=120, bbox_inches='tight')
print("Saved large_clone_xy_comparison.png")